# Model initiation

In [1]:
from extraction_tools import *
from models import *
from glob import glob

save_dir = r'D:\AIC_BUFFER'

keyframe_feature_extraction_model = SigLIP2Model(name="google/siglip2-giant-opt-patch16-384", device="cuda")
keyframe_extractor = KeyframeAndFeatureExtractor(
    keyframe_feature_extraction_model,
    save_dir
)

# transcription_feature_extraction_model = BGEM3Model(name="BAAI/bge-m3", device="cuda")
# transcription_extraction_model = FasterWhisperModel(name="turbo", device="cuda")
# transcription_extractor = TranscriptAndFeatureProcessor(
#     transcription_extraction_model,
#     transcription_feature_extraction_model,
#     save_dir
# )


[INFO] Loading quantized google/siglip2-giant-opt-patch16-384
[INFO] google/siglip2-giant-opt-patch16-384 is on cuda device


[transformers] Model config: bos_token_id must be `None` or an integer within the vocabulary (between 0 and 31999), got 49406. This may result in unexpected behavior.
[transformers] Model config: eos_token_id must be `None` or an integer within the vocabulary (between 0 and 31999), got 49407. This may result in unexpected behavior.


Loading weights:   0%|          | 0/1096 [00:00<?, ?it/s]

# Extract keyframes

In [ ]:
import subprocess
import numpy as np
import re

vid_paths = [
    # News
    r"D:\AIC26\vids\L21_V022.mp4",
    r"D:\AIC26\vids\L22_V011.mp4",
    # Bike racing
    r"D:\AIC26\vids\L23_V007.mp4",
    r"D:\AIC26\vids\L23_V024.mp4",
    # Dragon dance
    r"D:\AIC26\vids\L24_V002.mp4",
    r"D:\AIC26\vids\L24_V004.mp4",
    # Lion dance
    r"D:\AIC26\vids\L24_V015.mp4",
    r"D:\AIC26\vids\L24_V038.mp4",
    # Online class
    r"D:\AIC26\vids\L25_V060.mp4",
    r"D:\AIC26\vids\L25_V050.mp4",
    # Cooking
    r"D:\AIC26\vids\L26_V496.mp4",
    r"D:\AIC26\vids\L26_V290.mp4",
    # Tradition
    r"D:\AIC26\vids\L28_V002.mp4",
    r"D:\AIC26\vids\L29_V018.mp4",
    # Socials
    r"D:\AIC26\vids\L30_V083.mp4",
    r"D:\AIC26\vids\L30_V015.mp4",
]

# def extract_keyframes_ffmpeg(vid_path: str, shot_confidence_score: float = 0.1, min_keyframe_distance: float = 2):
#     select_str = f"select='isnan(prev_selected_t)+gt(scene,{shot_confidence_score})*gte(t-prev_selected_t,{min_keyframe_distance})',showinfo"
#     cmd = [
#         "ffmpeg",
#         "-hide_banner",
#         "-threads", "2",
#         "-i", vid_path,
#         "-v", "info",
#         "-skip_frame", "nokey",
#         "-vf", select_str,
#         "-fps_mode", "passthrough",
#         "-f", "rawvideo",
#         "-pix_fmt", "rgb24",
#         "-",
#     ]
#     result = subprocess.run(cmd, stdout=subprocess.PIPE, stderr=subprocess.PIPE)
#     stderr = result.stderr.decode("utf-8", errors="replace")
#     if result.returncode:
#         print(f"Error: {stderr}")
#         return [], []

#     stream_match = re.search(r"Stream #.* Video:.*", stderr)
#     if not stream_match: return [], []  
#     stream_line = stream_match.group(0)
#     fps_match = re.search(r"([\d\.]+)\s+fps", stream_line) or re.search(r"([\d\.]+)\s+tbr", stream_line)
#     res_match = re.search(r"(\d{2,5})x(\d{2,5})", stream_line)

#     fps = float(fps_match.group(1)) if fps_match else 0.0
#     width = int(res_match.group(1)) if res_match else None
#     height = int(res_match.group(2)) if res_match else None
#     time_orders = [float(t) for t in re.findall(r"pts_time:\s*([\d\.]+)", stderr)]
#     frame_orders = [time_order * fps for time_order in time_orders]
#     maps = [{
#         "vid_path": vid_path,
#         "frame_id": f"{(frame_id + 1):03d}",
#         "frame_order": frame_order,
#         "time_order": time_order,
#         "fps": fps,
#     } for frame_id, (frame_order, time_order) in enumerate(zip(frame_orders, time_orders))]

#     raw_frames = np.frombuffer(result.stdout, dtype=np.uint8)
#     imgs = raw_frames.reshape(-1, height, width, 3)

#     return imgs, maps

# imgs, maps = extract_keyframes_ffmpeg(vid_paths[0], 0.1, 2)

keyframe_extractor.extract_and_save_keyframes(
    vid_paths,
    feature_extraction_batch_size=16,
    shot_confidence_score=0.2,
    min_keyframe_distance=1,
    clustering_similarity_threshold=0.98,
    max_workers=4
)

Extracting keyframes from video: 100%|██████████| 16/16 [04:37<00:00, 17.35s/it]


In [2]:
print(imgs.shape)
print(len(maps))

(229, 720, 1280, 3)
229


# Extract transcripts

In [ ]:
vid_paths = glob("E:/AIC_VID_ONLY/vids/*/L0*.mp4") + glob("E:/AIC_VID_ONLY/vids/*/L1*.mp4")
map_paths = glob("D:/AIC_NEW_EXTRACTION/*/L0*.csv") + glob("D:/AIC_NEW_EXTRACTION/*/L1*.csv")

if len(vid_paths) != len(map_paths): raise RuntimeError(f"Mismatch videos and maps count: {len(vid_paths)} to {len(map_paths)}")

transcription_extractor.extract_and_save_transcript(
    vid_paths,
    map_paths,
    transcription_batch_size=16,
    transcription_language="vi",
    alignment_max_time_distance=10,
    feature_extraction_batch_size=32,
    max_workers=4
)

# Size compaction

In [ ]:
from extraction_tools import FeaturePackager, InfoPackager

feat_compactor = FeaturePackager(r"D:\AIC26_NO_VIDS\feats", r"D:\AIC26_NO_VIDS\full_feats")
feat_compactor.compact_and_save_features(max_workers=8)


info_compactor = InfoPackager(r"D:\AIC26_NO_VIDS\maps", r"D:\AIC26_NO_VIDS\full_maps")
info_compactor.add_info_to_map("frame", r"D:\AIC26_NO_VIDS\trans", [("transcript", "transcript")], max_workers=8)
info_compactor.add_info_to_map("video", r"D:\AIC26_NO_VIDS\mets", [("watch_url", "base_yt_link")], max_workers=8)
info_compactor.save_map()

# Full run

In [ ]:
import torch
import gc
from extraction_tools import *
from models import *
from glob import glob

save_dir = r"D:\AIC26_NO_VIDS"
vid_paths = glob(r"D:\AIC26\vids\*.mp4")

keyframe_extractor = KeyframeAndFeatureExtractor(
	TransNetV2Model(device="cuda"),
    SigLIP2Model(name="google/siglip2-so400m-patch14-384", device="cuda"),
    BEiT3Model("timm/beit3_large_patch16_224.pt", device="cuda"),
    save_dir
)

# Customize the parameters to fit your hardware setup
keyframe_extractor.extract_and_save_keyframes(
    vid_paths,
    perceptive_feature_extraction_batch_size=16,
    fused_feature_extraction_batch_size=64,
    shot_confidence_threshold=0.5,
    clustering_similarity_threshold=0.92,
    max_workers=4
)

# If your VRAM is not plenty, you can delete the unused KeyframeAndFeatureExtractor after use
del keyframe_extractor
gc.collect()
torch.cuda.empty_cache()

map_paths = glob(r"D:\AIC26_NO_VIDS\maps\*.csv")

transcript_extractor = TranscriptAndFeatureProcessor(
    FasterWhisperModel(name="turbo", device="cuda"),
    BGEM3Model(name="BAAI/bge-m3", device="cuda"),
    save_dir
)

# Customize the parameters to fit your hardware setup
transcript_extractor.extract_and_save_transcript(
    vid_paths,
    map_paths,
    transcription_batch_size=16,
    transcription_language="vi",
    alignment_max_time_distance=10,
    feature_extraction_batch_size=32,
    max_workers=4
)

# If your VRAM is not plenty, you can delete the unused TranscriptAndFeatureProcessor after use
del transcript_extractor
gc.collect()
torch.cuda.empty_cache()

feat_compactor = FeaturePackager(r"D:\AIC26_NO_VIDS\feats", r"D:\AIC26_NO_VIDS\full_feats")
feat_compactor.compact_and_save_features(max_workers=8)

info_compactor = InfoPackager(r"D:\AIC26_NO_VIDS\maps", r"D:\AIC26_NO_VIDS\full_maps")
info_compactor.add_info_to_map("frame", r"D:\AIC26_NO_VIDS\trans", [("transcript", "transcript")], max_workers=8)
info_compactor.add_info_to_map("video", r"D:\AIC26_NO_VIDS\mets", [("watch_url", "base_yt_link")], max_workers=8)
info_compactor.save_map()

[INFO] Loading quantized turbo
[INFO] turbo is on cuda device
[INFO] Loading quantized BAAI/bge-m3
[INFO] BAAI/bge-m3 is on cuda device


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 30 files:   0%|          | 0/30 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

Extracting transcripts from video: 100%|██████████| 873/873 [1:31:03<00:00,  6.26s/it]


# Test